# CCTV Safety — Stage 2 Fight / Non-Fight (X3D-S)

Kaggle notebook for the signed-off 300-clip SCFD pilot. This pilot has already been trained and evaluated; see `docs/project_status.md`. Stage 2 is a video-level temporal classifier using full-frame clips; it does not use Stage 1 YOLO weights.

**Training is disabled by default for any rerun.** Dataset checks are allowed, but model download, optimizer creation, training and test evaluation require new explicit owner authorization before changing TRAINING_APPROVED to True. For a rerun, create a Kaggle Notebook from the private SCFD dataset, select a GPU, and enable Internet for the pretrained X3D-S weights.

In [ ]:
from pathlib import Path
import os, json, csv, hashlib, random
from collections import Counter

TRAINING_APPROVED = False  # Keep False until explicit owner authorization.
DATA_ROOT_OVERRIDE = None  # Optional absolute Kaggle input path to scfd_pilot.
RUN_ROOT = Path('/kaggle/working/stage2_runs')
RUN_NAME = 'scfd_x3d_s_full_frame'
RESUME_CHECKPOINT = None  # Set to attached prior output's last.pt to resume.
EXPECTED_SPLITS = {'train': 210, 'val': 45, 'test': 45}
EXPECTED_CLASSES = {'train': {0: 105, 1: 105}, 'val': {0: 23, 1: 22}, 'test': {0: 22, 1: 23}}
CLASS_NAMES = ['non_fight', 'fight']
FRAMES, SIZE, BATCH_SIZE, WORKERS = 13, 224, 8, 2
MAX_EPOCHS, PATIENCE, LR, SEED = 30, 6, 1e-4, 42
print('Training gate:', TRAINING_APPROVED)

## Validate the attached SCFD package

Expected contents: scfd_pilot_manifest.json, train/val/test manifests, and flat split folders with MP4 clips. The gate verifies all approved rows, labels, exact counts, hashes, decodability and group-level split isolation.

In [ ]:
def locate_dataset():
    if DATA_ROOT_OVERRIDE:
        p = Path(DATA_ROOT_OVERRIDE)
        if (p / 'scfd_pilot_manifest.json').is_file(): return p
        raise FileNotFoundError(f'Manifest not found under {p}')
    found = set()
    for base in (Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()):
        if base.exists(): found.update(p.parent.resolve() for p in base.rglob('scfd_pilot_manifest.json'))
    if len(found) != 1: raise RuntimeError(f'Expected one SCFD package; found {len(found)}. Set DATA_ROOT_OVERRIDE: {sorted(map(str, found))[:8]}')
    return next(iter(found))

DATA_ROOT = locate_dataset()
manifest_path = DATA_ROOT / 'scfd_pilot_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['total_clips'] == 300 and manifest['splits'] == EXPECTED_SPLITS
assert manifest['class_mapping'] == {'0': 'non_fight', '1': 'fight'}
import cv2
rows_by_split, groups, hashes = {}, {}, set()
for split in ('train', 'val', 'test'):
    with (DATA_ROOT / f'{split}_manifest.csv').open(encoding='utf-8-sig', newline='') as f: rows = list(csv.DictReader(f))
    assert len(rows) == EXPECTED_SPLITS[split]
    counts, split_groups = Counter(), set()
    for row in rows:
        y = int(row['label_id']); assert y in (0, 1) and row['binary_source_label'] == CLASS_NAMES[y]
        assert row['split'] == split and row['qa_status'] == 'OWNER_HUMAN_QA_APPROVED' and row['owner_approved'].lower() == 'true'
        split_groups.add(row['scene_group_id'])
        p = DATA_ROOT / split / f"{row['clip_id']}.mp4"
        if not p.is_file(): p = DATA_ROOT / split / Path(row['relative_path']).name
        assert p.is_file() and p.stat().st_size == int(row['byte_size']), f'Missing/size mismatch: {p}'
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        assert digest == row['sha256'] and digest not in hashes, f'Hash mismatch or duplicate: {p}'
        hashes.add(digest)
        cap = cv2.VideoCapture(str(p)); ok, frame = cap.read(); cap.release()
        assert ok and frame is not None and frame.size, f'Undecodable video: {p}'
        row['_path'] = str(p); counts[y] += 1
    assert dict(counts) == EXPECTED_CLASSES[split], (split, dict(counts))
    rows_by_split[split], groups[split] = rows, split_groups
assert not (groups['train'] & groups['val'] or groups['train'] & groups['test'] or groups['val'] & groups['test']), 'Scene-group leakage across splits'
assert len(hashes) == 300
print('PASS:', DATA_ROOT, '| 300 approved/decodeable/hash-verified clips | no group leakage')
print('Dataset manifest:', manifest_path)

## Training gate and model setup

Only continue after explicit authorization. X3D-S uses 13 uniformly sampled frames; resize preserves aspect ratio and pads to 224×224. Kinetics channel normalization is used. Set Kaggle Internet on for first-time pretrained model download.

In [ ]:
if not TRAINING_APPROVED:
    raise RuntimeError('STOP: Stage 2 training not authorized. No pretrained model, optimizer, training, or test evaluation executed.')
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorchvideo', 'fvcore', 'iopath'])
import torch, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, balanced_accuracy_score
assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator.'
device = torch.device('cuda:0')

class ClipDataset(Dataset):
    def __init__(self, rows, augment=False): self.rows, self.augment = rows, augment
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]; cap = cv2.VideoCapture(row['_path']); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if not cap.isOpened() or n < 1: cap.release(); raise RuntimeError(f"Cannot decode {row['_path']}")
        frames = []
        for idx in np.linspace(0, n-1, FRAMES).round().astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx)); ok, bgr = cap.read()
            if not ok: cap.release(); raise RuntimeError(f"Frame decode failed: {row['_path']}")
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); h,w = rgb.shape[:2]
            scale = SIZE / max(h,w); nh,nw = max(1,round(h*scale)),max(1,round(w*scale))
            im = cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA); canvas = np.zeros((SIZE,SIZE,3),np.uint8)
            y,x = (SIZE-nh)//2,(SIZE-nw)//2; canvas[y:y+nh,x:x+nw] = im; frames.append(canvas)
        cap.release()
        if self.augment and random.random() < .5: frames = [np.ascontiguousarray(f[:,::-1]) for f in frames]
        a = np.stack(frames).astype(np.float32)/255.; a = (a-.45)/.225
        return torch.from_numpy(a).permute(3,0,1,2).contiguous(), torch.tensor(int(row['label_id']))

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
train_ds, val_ds, test_ds = (ClipDataset(rows_by_split[s], s=='train') for s in ('train','val','test'))
train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=WORKERS,pin_memory=True)
val_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=WORKERS,pin_memory=True)
test_loader = DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=WORKERS,pin_memory=True)
model = torch.hub.load('facebookresearch/pytorchvideo:main','x3d_s',pretrained=True)
head = model.blocks[-1]; assert hasattr(head,'proj') and hasattr(head.proj,'in_features')
head.proj = nn.Linear(head.proj.in_features,2); head.activation = None
model = model.to(device)
with torch.no_grad(): assert model(torch.zeros(2,3,FRAMES,SIZE,SIZE,device=device)).shape == (2,2)
print('X3D-S ready on', torch.cuda.get_device_name(0))

## Train and checkpoint

The last checkpoint is atomically written each epoch and contains model/optimizer/scheduler/scaler state, best validation loss, early-stop counter, history, RNG states and manifest hash. Save a Kaggle notebook version with output before session expiry. For resume, attach that output as an input, set RESUME_CHECKPOINT to last.pt, and keep manifest/preprocessing unchanged.

In [ ]:
import torch.nn.functional as F
RUN_DIR = RUN_ROOT / RUN_NAME; RUN_DIR.mkdir(parents=True, exist_ok=True)
LAST, BEST = RUN_DIR/'last.pt', RUN_DIR/'best.pt'
manifest_hash = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
criterion=nn.CrossEntropyLoss(); optimizer=AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=2); scaler=GradScaler('cuda')
start,best_loss,bad,history=0,float('inf'),0,[]
if RESUME_CHECKPOINT:
    ck=torch.load(RESUME_CHECKPOINT,map_location='cpu',weights_only=False)
    assert ck['manifest_hash']==manifest_hash and ck['frames']==FRAMES and ck['size']==SIZE, 'Checkpoint/data config mismatch'
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
    start,best_loss,bad,history=ck['epoch']+1,ck['best_loss'],ck['bad'],ck['history']
    random.setstate(ck['py_rng']); np.random.set_state(ck['np_rng']); torch.set_rng_state(ck['torch_rng'])
    if ck.get('cuda_rng'): torch.cuda.set_rng_state_all(ck['cuda_rng'])
else:
    assert not LAST.exists() and not BEST.exists(), f'Run exists: {RUN_DIR}; choose new RUN_NAME or resume.'

def evaluate(loader):
    model.eval(); loss_sum=0.; ys=[]; ps=[]
    with torch.inference_mode():
        for x,y in loader:
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            with autocast('cuda',dtype=torch.float16): z=model(x)
            loss_sum += criterion(z.float(),y).item()*len(y); ys += y.cpu().tolist(); ps += z.argmax(1).cpu().tolist()
    return loss_sum/len(loader.dataset),f1_score(ys,ps,average='macro'),balanced_accuracy_score(ys,ps),ys,ps

for epoch in range(start,MAX_EPOCHS):
    model.train(); total=0.; count=0
    for x,y in tqdm(train_loader,desc=f'Epoch {epoch+1}/{MAX_EPOCHS}'):
        x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True); optimizer.zero_grad(set_to_none=True)
        with autocast('cuda',dtype=torch.float16): z=model(x); loss=criterion(z,y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); total+=loss.item()*len(y); count+=len(y)
    vl,vf,vb,_,_=evaluate(val_loader); scheduler.step(vl)
    history.append({'epoch':epoch+1,'train_loss':total/count,'val_loss':vl,'val_macro_f1':vf,'val_balanced_accuracy':vb,'lr':optimizer.param_groups[0]['lr']})
    improved=vl<best_loss
    if improved: best_loss,bad=vl,0
    else: bad+=1
    state={'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'best_loss':best_loss,'bad':bad,'history':history,'manifest_hash':manifest_hash,'frames':FRAMES,'size':SIZE,'py_rng':random.getstate(),'np_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()}
    tmp=LAST.with_suffix('.tmp'); torch.save(state,tmp); os.replace(tmp,LAST)
    if improved:
        tmp=BEST.with_suffix('.tmp'); torch.save(state,tmp); os.replace(tmp,BEST)
    (RUN_DIR/'training_history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
    print(history[-1])
    if bad>=PATIENCE: print('Early stop: validation loss patience reached.'); break
print('Checkpoints:', LAST, BEST)

## One-time held-out test evaluation

Run only after selecting the best validation checkpoint. Do not use test metrics to tune the model. Save outputs as a Kaggle notebook version; an unversioned working session may disappear.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
assert TRAINING_APPROVED and BEST.is_file(), 'Training gate is closed or best checkpoint is missing.'
best=torch.load(BEST,map_location='cpu',weights_only=False)
assert best['manifest_hash']==manifest_hash
model.load_state_dict(best['model']); model.to(device)
_,_,_,y_true,y_pred=evaluate(test_loader)
report={'model':'X3D-S','task':'fight_vs_non_fight','dataset_version':manifest['dataset_version'],'manifest_sha256':manifest_hash,'test_clips':len(test_ds),'class_names':CLASS_NAMES,'macro_f1':f1_score(y_true,y_pred,average='macro'),'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),'classification_report':classification_report(y_true,y_pred,target_names=CLASS_NAMES,output_dict=True,zero_division=0),'confusion_matrix_true_rows_pred_columns':confusion_matrix(y_true,y_pred,labels=[0,1]).tolist()}
(RUN_DIR/'test_metrics.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps(report,indent=2)); print('Artifacts:',sorted(p.name for p in RUN_DIR.iterdir()))